# Scaling Laws

This notebook contains a set of experiments to discover scaling laws on a character-tokenized dataset and a small language model based on the encoder-only transformer architecture.

References:

- https://arxiv.org/abs/2001.08361

- https://arxiv.org/pdf/2203.15556

- https://github.com/BenAgro314/MinChilla?tab=readme-ov-file

## Utils

### Model definition

In [11]:
import torch.nn as nn
import torch
import math
from dataclasses import dataclass


torch.manual_seed(1233)


@dataclass
class ModelConfig:
    vocab_size: int = 64
    n_embedding: int = 1024
    n_layers: int = 2
    n_heads: int = 4
    context_window: int = 128


class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding from "Attention is All You Need".
    """
    def __init__(self, n_embedding, context_window):
        super().__init__()
        self.pe = torch.zeros(context_window, n_embedding)
        position = torch.arange(0, context_window, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, n_embedding, 2).float() * (-math.log(10000.0) / n_embedding)
        )
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = self.pe.unsqueeze(0)  # shape: (1, context_window, n_embedding)

    def forward(self, x):
        """
        x: shape [batch_size, seq_len, n_embedding]
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :].to(x.device)
    

class SmallTransformer(nn.Module):
    """
    Small Transformer Based Model
    """

    def __init__(self, model_config: ModelConfig):

        super().__init__()
        
        # model hyper-parameters
        self.vocab_size = model_config.vocab_size
        self.n_embedding = model_config.n_embedding
        self.n_layers = model_config.n_layers
        self.n_heads = model_config.n_heads
        self.context_window = model_config.context_window

        # embeddings
        self.token_embedding = nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.n_embedding)
        self.positional_embedding = PositionalEncoding(context_window=self.context_window, n_embedding=self.n_embedding)

        # N transforme layers
        dim_feedforward = self.n_embedding * 2
        encoder_layer = nn.TransformerEncoderLayer(nhead=self.n_heads, d_model=self.n_embedding, dim_feedforward=dim_feedforward, batch_first=True)
        layer_norm = nn.LayerNorm(self.n_embedding)
        self.transformer_blocks = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=self.n_layers, norm=layer_norm)

        # Linear Head
        self.lm_head = nn.Linear(self.n_embedding, self.vocab_size)

    def forward(self, x):
        """
        x: shape [batch_size, seq_len, n_embedding]
        """
        out = self.token_embedding(x)
        out = self.positional_embedding(out)
        out = self.transformer_blocks(out)
        out = self.lm_head(out)
        return out

### Data Processing

In [12]:
import torch 
for i in iter(torch.randperm(len("gola")).tolist()):
    print(i)

0
1
2
3


In [13]:
from torch.utils.data import Dataset, DataLoader, Sampler


class TokenLimitedSampler(Sampler):
    def __init__(self, data_source, num_tokens: int = None):
        self.data_source = data_source
        self.num_tokens = num_tokens if num_tokens is not None else len(self.data_source)

    def __iter__(self):
        return iter(torch.randperm(len(self.data_source))[:self.num_tokens].tolist())

    def __len__(self):
        return self.num_tokens


class CharacterDataset(Dataset):
    """
    Creates a token-level dataset by slicing a single long token tensor.
    Returns (x, y) pairs where x and y are arrays of integer IDs for tokens.
    """
    def __init__(self, tokens, seq_len):
        super().__init__()
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        # We'll generate slices up to len(tokens)-seq_len
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        x = self.tokens[idx : idx + self.seq_len]
        y = self.tokens[idx + 1 : idx + self.seq_len + 1]
        return x, y


class CharacterTokenizer:

    def __init__(self):
        self.char2idx = {}
        self.idx2char = {}
        self._vocab_size = None

    def train(self, train_data):
        """
        Builds char2idx and idx2char mappings given a list of strings.
        """
        # Collect all characters
        unique_chars = set("".join([item['text'] for item in train_data]))
        # Ensure a padding character exists
        unique_chars.add(' ')  # Adding space as padding if not present
        # Sort for reproducibility
        unique_chars = sorted(list(unique_chars))
        self.char2idx = {ch: i for i, ch in enumerate(unique_chars)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}

        self._vocab_size = len(self.char2idx)

        print(f"Built vocabulary with {self._vocab_size} characters.")
    
    @property
    def vocab_size(self):
        return self._vocab_size
    
    def encode(self, text):
        tokens = torch.tensor([self.char2idx.get(ch, self.char2idx[' ']) for ch in text], dtype=torch.long)
        return tokens
    
    def decode(self, tokens):
        text = "".join([self.idx2char.get(tok, self.idx2char[self.char2idx[' ']]) for tok in tokens])
        return text

### Training & Eval utils

In [39]:

from torch.optim import AdamW
import torch.nn.functional as F


def criterion(y, targets):
    """
    Compute Cross Entropy
    """
    loss = F.cross_entropy(y, targets)
    return loss


from tqdm import tqdm


def train_one_epoch(optimizer, model, train_data_loader: DataLoader, epoch: int, writer):

    # activate train mode
    model.train()

    progress_bar = tqdm(enumerate(train_data_loader), total=len(train_data_loader), desc=f"Epoch {epoch+1} [Train]")

    epoch_loss = 0.0

    for batch_idx, (x, y) in progress_bar:

        # remove old gradients
        optimizer.zero_grad()

        # compute loss
        logits = model(x)
        train_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        epoch_loss += train_loss

        # compute gradients
        train_loss.backward()

        # back-propagate
        optimizer.step()

        writer.add_scalar("Loss/Train_iter", train_loss.item(), epoch * len(train_data_loader) + batch_idx)

    epoch_loss = epoch_loss / len(train_data_loader)
    writer.add_scalar("Loss/Train_Epoch", epoch_loss, epoch)  


@torch.no_grad()
def evaluate(model, validation_data_loader, epoch: int, writer):

    # activate eval mode
    model.eval()

    progress_bar = tqdm(enumerate(validation_data_loader), total=len(validation_data_loader), desc=f"Epoch {epoch+1} [Validation]")

    epoch_loss = 0.0

    for batch_idx, (x, y) in progress_bar:

        # compute loss
        logits = model(x)
        val_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        epoch_loss += val_loss

        writer.add_scalar("Loss/Val_iter", val_loss.item(), epoch * len(validation_data_loader) + batch_idx)
    
    epoch_loss = epoch_loss / len(validation_data_loader)
    writer.add_scalar("Loss/Val_Epoch", epoch_loss, epoch)


In [40]:
def get_param_count(model: SmallTransformer):    
    return sum([p.numel() for p in model.parameters()])

def convert_flops_to_tokens(flops: int, param_count: int) -> int:
    num_tokens = int(flops / (6 * param_count))
    return num_tokens

## Main Code

### Load, Tokenize

In [41]:
from datasets import load_dataset

# -----------------------
train_data = load_dataset("roneneldan/TinyStories", split="train[:5]")
validation_data = load_dataset("roneneldan/TinyStories", split="validation[:2]")

# -----------------------
tokenizer = CharacterTokenizer()
tokenizer.train(train_data)

train_tokens = torch.concat([tokenizer.encode(item['text']) for item in train_data])
validation_tokens = torch.concat([tokenizer.encode(item['text']) for item in validation_data])

Built vocabulary with 49 characters.


### Model

In [42]:
# -----------------------
model_config = ModelConfig(vocab_size=tokenizer.vocab_size, n_embedding=256)
print(model_config)
model = SmallTransformer(model_config)
print(f"Number of model parameters: {sum([l.numel() for l in model.parameters()])/1e6:.2f}")


ModelConfig(vocab_size=49, n_embedding=256, n_layers=2, n_heads=4, context_window=128)
Number of model parameters: 1.08


### Training Loop

In [ ]:
# -----------------------
from torch.utils.tensorboard import SummaryWriter


@dataclass
class TrainingConfig:
    max_epochs: int = 1
    lr: float = 3e-4
    batch_size: int = 128

training_config = TrainingConfig()
optimizer = AdamW(model.parameters(), lr=training_config.lr)

# -----------------------
train_dataset = CharacterDataset(train_tokens, seq_len=model_config.context_window)
validation_dataset = CharacterDataset(validation_tokens, seq_len=model_config.context_window)
train_data_loader = DataLoader(
    train_dataset,
    batch_size=training_config.batch_size,
    sampler=TokenLimitedSampler(train_dataset)
)
validation_data_loader = DataLoader(
    validation_dataset,
    batch_size=training_config.batch_size,
    sampler=TokenLimitedSampler(validation_dataset)
)

# -----------------------
log_file_name = "experiments/dummy_experiment/logs"
writer = SummaryWriter(log_file_name)

writer.add_scalar("Model/Total Parameters", get_param_count(model), 0)


for epoch in range(training_config.max_epochs):

    train_one_epoch(
        optimizer=optimizer,
        model=model,
        train_data_loader=train_data_loader,
        epoch=epoch,
        writer=writer,
    )

    evaluate(
        model=model,
        validation_data_loader=validation_data_loader,
        epoch=epoch,
        writer=writer
    )


Epoch 1 [Validation]: 100%|██████████| 12/12 [00:02<00:00,  5.60it/s]


In [59]:
# !tensorboard --logdir ./experiments

Extract logged data from tensorboard

In [ ]:
import os
from pathlib import Path
from typing import Iterable, Optional, Sequence

import pandas as pd

from tensorboard.backend.event_processing import event_accumulator


def extract_scalars_from_tb(
    logdir: str | os.PathLike,
    tags: Optional[Sequence[str]] = None,
    recursive: bool = True,
) -> pd.DataFrame:
    """
    Extract scalar time series from TensorBoard event files under `logdir`.

    Args:
        logdir: Path that contains TensorBoard event files (e.g., outputs/exp/logs).
        tags: If provided, only extract these scalar tags (exact match).
              If None, extract all scalar tags found.
        recursive: Whether to scan subdirectories for event files.

    Returns:
        DataFrame with columns: ["run", "tag", "step", "wall_time", "value"]
    """
    logdir = Path(logdir)

    # TensorBoard expects a "run directory" that contains one or more event files.
    # We treat each directory that has event files as a separate "run".
    pattern = "**/events.out.tfevents.*" if recursive else "events.out.tfevents.*"
    event_files = list(logdir.glob(pattern))

    if not event_files:
        raise FileNotFoundError(f"No TensorBoard event files found under: {logdir}")

    run_dirs = sorted({f.parent for f in event_files})

    rows: list[dict] = []

    for run_dir in run_dirs:
        # Load events for this run directory
        ea = event_accumulator.EventAccumulator(
            str(run_dir),
            size_guidance={event_accumulator.SCALARS: 0},  # 0 = load all scalars
        )
        ea.Reload()

        available_tags = ea.Tags().get("scalars", [])
        if not available_tags:
            continue

        wanted_tags: Iterable[str]
        if tags is None:
            wanted_tags = available_tags
        else:
            # Keep only tags that exist in this run
            wanted_tags = [t for t in tags if t in available_tags]

        for tag in wanted_tags:
            for ev in ea.Scalars(tag):
                rows.append(
                    {
                        "run": str(run_dir.relative_to(logdir)),
                        "tag": tag,
                        "step": ev.step,
                        "wall_time": ev.wall_time,
                        "value": ev.value,
                    }
                )

    if not rows:
        raise RuntimeError(
            f"No scalar data found. Check that scalars were logged and logdir is correct: {logdir}"
        )

    df = pd.DataFrame(rows).sort_values(["run", "tag", "step"]).reset_index(drop=True)
    return df


# Example: point to the directory you pass to SummaryWriter(log_dir=...)
LOGDIR = "experiments"  # or "outputs/experiment-.../logs"

# Put your loss tags here. If you leave tags=None, it extracts all scalar tags.
LOSS_TAGS = None # [
#     "Loss/Train_iter",
#     "Loss/Train_iter_smoothed",
#     "Loss/Val_iter",
#     "Loss/Val_Epoch",
# ]

df = extract_scalars_from_tb(LOGDIR, tags=LOSS_TAGS, recursive=True)
df

,run,tag,step,wall_time,value
0,dummy_experiment/logs,Loss/Train_Epoch,0,1.768929e+09,2.816654e+00
1,dummy_experiment/logs,Loss/Train_iter,0,1.768929e+09,4.093327e+00
2,dummy_experiment/logs,Loss/Train_iter,1,1.768929e+09,3.725681e+00
3,dummy_experiment/logs,Loss/Train_iter,2,1.768929e+09,3.440052e+00
4,dummy_experiment/logs,Loss/Train_iter,3,1.768929e+09,3.254148e+00
5,dummy_experiment/logs,Loss/Train_iter,4,1.768929e+09,3.150628e+00
6,dummy_experiment/logs,Loss/Train_iter,5,1.768929e+09,3.059058e+00
7,dummy_experiment/logs,Loss/Train_iter,6,1.768929e+09,2.958805e+00
8,dummy_experiment/logs,Loss/Train_iter,7,1.768929e+09,2.918975e+00
9,dummy_experiment/logs,Loss/Train_iter,8,1.768929e+09,2.877086e+00


## N Experiments

In [ ]:
from datasets import load_dataset

# -----------------------
train_data = load_dataset("roneneldan/TinyStories", split="train[:5]")
validation_data = load_dataset("roneneldan/TinyStories", split="validation[:2]")

# -----------------------
tokenizer = CharacterTokenizer()
tokenizer.train(train_data)

train_tokens = torch.concat([tokenizer.encode(item['text']) for item in train_data])
validation_tokens = torch.concat([tokenizer.encode(item['text']) for item in validation_data])

train_dataset = CharacterDataset(train_tokens, seq_len=model_config.context_window)
validation_dataset = CharacterDataset(validation_tokens, seq_len=model_config.context_window)


# -----------------------
# N Experiments

flop_counts = [
    1e15, # 1 PetaFLOP
    3e15, 
    6e15, 
    1e16, 
    3e16,  # 30 PetaFLOPs
]
model_sizes = range(128, 64*15, 64)
experiment_configs = []


for flops in flop_counts:

    for n_embedding in model_sizes:

        # -----------------------
        # define model config
        n_heads = max(n_embedding // 32, 1)
        n_layers = max(n_embedding // 32, 2)
    
        model_config = ModelConfig(
            vocab_size=tokenizer.vocab_size, 
            n_embedding=n_embedding,
            n_layers=n_layers,
            n_heads=n_heads
        )

        model = SmallTransformer(model_config)
        params_count = get_param_count(model)
        
        # -----------------------
        # Training config
        @dataclass
        class TrainingConfig:
            max_epochs: int = 1
            lr: float = 3e-4
            batch_size: int = 128

        training_config = TrainingConfig()
        optimizer = AdamW(model.parameters(), lr=training_config.lr)

        # -----------------------
        # Training data
        num_tokens = convert_flops_to_tokens(flops=flops, param_count=params_count)
        train_iters = num_tokens / (training_config.batch_size * model_config.context_window)

        print(f"petaflops={flops/1e15:.2e} | params={params_count} (n_embedding={n_embedding}, n_heads={n_heads}, n_layers={n_layers}) | tokens={num_tokens:.2e}")
        exp_config = {'petaflops': flops/1e15, "params": params_count}| ModelConfig().__dict__
        experiment_configs.append(exp_config)

        train_data_loader = DataLoader(
            train_dataset,
            batch_size=training_config.batch_size,
            sampler=TokenLimitedSampler(train_dataset, num_tokens=num_tokens)
        )
        validation_data_loader = DataLoader(
            validation_dataset,
            batch_size=training_config.batch_size,
            sampler=TokenLimitedSampler(train_dataset, num_tokens=num_tokens)
        )

        # -----------------------
        log_file_name = f"experiments/experiment_{int(flops/1e15)}_{n_embedding}/logs"
        writer = SummaryWriter(log_file_name)

        writer.add_scalar("Model/Total Parameters", get_param_count(model), 0)

        for epoch in range(training_config.max_epochs):

            train_one_epoch(
                optimizer=optimizer,
                model=model,
                train_data_loader=train_data_loader,
                epoch=epoch,
                writer=writer,
            )

            evaluate(
                model=model,
                validation_data_loader=validation_data_loader,
                epoch=epoch,
                writer=writer
        )


Built vocabulary with 49 characters.
petaflops=1.00e+00 | params=542769 (n_embedding=128, n_heads=4, n_layers=4) | tokens=3.07e+08
petaflops=1.00e+00 | params=1801393 (n_embedding=192, n_heads=6, n_layers=6) | tokens=9.25e+07
petaflops=1.00e+00 | params=4242481 (n_embedding=256, n_heads=8, n_layers=8) | tokens=3.93e+07
petaflops=1.00e+00 | params=8259249 (n_embedding=320, n_heads=10, n_layers=10) | tokens=2.02e+07
petaflops=1.00e+00 | params=14244913 (n_embedding=384, n_heads=12, n_layers=12) | tokens=1.17e+07
petaflops=1.00e+00 | params=22592689 (n_embedding=448, n_heads=14, n_layers=14) | tokens=7.38e+06
petaflops=1.00e+00 | params=33695793 (n_embedding=512, n_heads=16, n_layers=16) | tokens=4.95e+06
petaflops=1.00e+00 | params=47947441 (n_embedding=576, n_heads=18, n_layers=18) | tokens=3.48e+06
petaflops=1.00e+00 | params=65740849 (n_embedding=640, n_heads=20, n_layers=20) | tokens=2.54e+06
petaflops=1.00e+00 | params=87469233 (n_embedding=704, n_heads=22, n_layers=22) | tokens=1.9

In [32]:
import pandas as pd

pd.DataFrame(experiment_configs)

,petaflops,params,vocab_size,n_embedding,n_layers,n_heads,context_window
0,1.0,4242481,64,1024,2,4,128
1,1.0,8259249,64,1024,2,4,128
2,1.0,14244913,64,1024,2,4,128
3,1.0,22592689,64,1024,2,4,128
4,1.0,33695793,64,1024,2,4,128
...,...,...,...,...,...,...,...
60,30.0,113525809,64,1024,2,4,128
61,30.0,144303793,64,1024,2,4,128
62,30.0,180196401,64,1024,2,4,128
63,30.0,221596849,64,1024,2,4,128
